# 04 — MIDAS Regression (R / midasr)
Mixed Data Sampling: forecast daily silver returns using monthly macro variables.

Models implemented (simple → complex):
1. **U-MIDAS** — unrestricted OLS on stacked monthly lags (baseline)
2. **ADL-MIDAS** — U-MIDAS + AR(1) term on the daily target
3. **Beta-MIDAS (single)** — Beta polynomial weights, one predictor at a time
4. **Almon-MIDAS (single)** — Exponential Almon weights, one predictor at a time
5. **Multi-predictor Beta-MIDAS** — all macro vars, each with own Beta weights
6. **MIDAS + Sentiment** — weekly Reddit sentiment as additional MIDAS predictor


## Setup

In [ ]:
library(midasr)
library(ggplot2)
library(dplyr)
library(tidyr)

# ── Evaluation metrics (match Python notebooks exactly) ───────────────────
rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm=TRUE))
mae  <- function(a, p) mean(abs(a - p),      na.rm=TRUE)
da   <- function(a, p) mean(sign(a) == sign(p), na.rm=TRUE)
wda  <- function(a, p) {
  ok <- sign(a) == sign(p)
  sum(abs(a) * ok, na.rm=TRUE) / sum(abs(a), na.rm=TRUE)
}
eval_model <- function(name, actual, pred) {
  cat(sprintf("%-30s  RMSE=%.6f  MAE=%.6f  DA=%.3f  WDA=%.3f\n",
              name, rmse(actual,pred), mae(actual,pred),
              da(actual,pred), wda(actual,pred)))
  data.frame(model=name, rmse=rmse(actual,pred), mae=mae(actual,pred),
             dir_acc=da(actual,pred), wda=wda(actual,pred))
}

cat("midasr:", as.character(packageVersion("midasr")), "\n")


## 1. Load data

In [ ]:
train <- read.csv("../data/processed/train.csv", row.names=1)
test  <- read.csv("../data/processed/test.csv",  row.names=1)
macro <- read.csv("../data/raw/monthly_macro.csv", row.names=1)

train$Date <- as.Date(rownames(train))
test$Date  <- as.Date(rownames(test))
macro$Date <- as.Date(rownames(macro))

TARGET     <- "silver_return"
MACRO_VARS <- intersect(c("cpi","fed_funds","ind_prod","real_rates"), colnames(macro))
N_LAGS     <- 3   # monthly lags to include (= roughly one quarter)

cat("Train:", nrow(train), " Test:", nrow(test), "\n")
cat("Macro vars:", paste(MACRO_VARS, collapse=", "), "\n")


## 2. Helper functions

`build_lag_matrix()` — for each daily date, look back and grab the last
`n_lags` monthly observations of a given variable. Returns an (N × K) matrix
where row i is the lag vector available on day i.

The two weight functions implement the standard MIDAS polynomials:

**Beta weights** — hump-shaped or monotone decay, controlled by (θ₁, θ₂ > 0):

$$w_k(\theta_1,\theta_2) = \frac{f(k/K;\,\theta_1,\theta_2)}{\sum_{j=1}^{K}f(j/K;\,\theta_1,\theta_2)}, \qquad f(x;\,\theta_1,\theta_2) = x^{\theta_1-1}(1-x)^{\theta_2-1}$$

$f$ is the Beta density kernel. Equal parameters → symmetric hump; $\theta_1 < \theta_2$ → more weight on recent lags; $\theta_1 = \theta_2 = 1$ → uniform weights.

**Normalised exponential Almon weights** — controlled by (θ₁, θ₂):

$$w_k(\theta_1,\theta_2) = \frac{\exp(\theta_1 k + \theta_2 k^2)}{\sum_{j=1}^{K}\exp(\theta_1 j + \theta_2 j^2)}$$

The exponential ensures positivity; normalisation ensures weights sum to one. θ₂ < 0 gives a decaying profile; θ₂ > 0 gives a hump.

Both parameterisations replace K free OLS coefficients with just 2 shape parameters.

In [ ]:
# Build (N x K) lag matrix for one variable
build_lag_matrix <- function(daily_df, macro_df, var, n_lags) {
  mat <- matrix(NA_real_, nrow=nrow(daily_df), ncol=n_lags)
  for (i in seq_len(nrow(daily_df))) {
    past <- na.omit(macro_df[[var]][macro_df$Date <= daily_df$Date[i]])
    if (length(past) >= n_lags)
      mat[i, ] <- rev(tail(past, n_lags))   # lag1 = most recent
  }
  colnames(mat) <- paste0(var, "_lag", seq_len(n_lags))
  mat
}

# Normalised Beta polynomial weights  (θ₁, θ₂ > 0)
nbeta_w <- function(k, theta) {
  x <- k / (max(k) + 1)
  w <- x^(theta[1]-1) * (1-x)^(theta[2]-1)
  w <- pmax(w, 1e-10)
  w / sum(w)
}

# Normalised exponential Almon weights
nealmon_w <- function(k, theta) {
  w <- exp(theta[1]*k + theta[2]*k^2)
  w / sum(w)
}

# Given a lag matrix and weight function, compute weighted single column
apply_weight_fn <- function(lag_mat, weight_fn, theta) {
  w <- weight_fn(seq_len(ncol(lag_mat)), theta)
  lag_mat %*% w
}

cat("Helper functions defined.\n")


## 3. Build feature matrices for all models

In [8]:
# Build lag matrices for every macro variable
lag_train <- lapply(MACRO_VARS, build_lag_matrix, daily_df=train, macro_df=macro, n_lags=N_LAGS)
lag_test  <- lapply(MACRO_VARS, build_lag_matrix, daily_df=test,  macro_df=macro, n_lags=N_LAGS)
names(lag_train) <- names(lag_test) <- MACRO_VARS

y_train <- train[[TARGET]]
y_test  <- test[[TARGET]]

# Stack all lag matrices into a single design matrix for U-MIDAS / ADL-MIDAS
X_train_u <- do.call(cbind, lag_train)
X_test_u  <- do.call(cbind, lag_test)

# Mask rows with any NA
ok_train <- complete.cases(X_train_u) & !is.na(y_train)
ok_test  <- complete.cases(X_test_u)  & !is.na(y_test)

cat("Train rows (complete):", sum(ok_train), "\n")
cat("Test  rows (complete):", sum(ok_test),  "\n")


Train rows (complete): 1717 
Test  rows (complete): 500 


## 4. U-MIDAS — Unrestricted MIDAS (OLS baseline)

Each monthly lag enters as a free OLS coefficient.
With 4 variables × 3 lags = 12 regressors plus an intercept.


In [ ]:
df_tr <- data.frame(target=y_train[ok_train], X_train_u[ok_train, ])
df_te <- data.frame(X_test_u[ok_test, ])

ols_u <- lm(target ~ ., data=df_tr)
preds_umidas <- predict(ols_u, newdata=df_te)

res_umidas <- eval_model("U-MIDAS", y_test[ok_test], preds_umidas)


## 5. ADL-MIDAS — Autoregressive Distributed Lag MIDAS

Adds a one-day lag of silver return (AR term) to U-MIDAS.
This is the most commonly used MIDAS variant in practice because the AR term
absorbs any residual autocorrelation the macro lags don't capture.


In [ ]:
# AR(1) term: y_{t-1}
ar1_train <- c(NA, y_train[-length(y_train)])
ar1_test  <- c(NA, y_test[-length(y_test)])

ok_adl_tr <- ok_train & !is.na(ar1_train)
ok_adl_te <- ok_test  & !is.na(ar1_test)

df_adl_tr <- data.frame(target=y_train[ok_adl_tr],
                         ar1=ar1_train[ok_adl_tr],
                         X_train_u[ok_adl_tr, ])
df_adl_te <- data.frame(ar1=ar1_test[ok_adl_te],
                         X_test_u[ok_adl_te, ])

ols_adl <- lm(target ~ ., data=df_adl_tr)
preds_adl <- predict(ols_adl, newdata=df_adl_te)

res_adl <- eval_model("ADL-MIDAS", y_test[ok_adl_te], preds_adl)


## 6. Beta-MIDAS — single predictor, all macro variables

For each macro variable separately, optimise the two Beta polynomial shape
parameters (θ₁, θ₂) that minimise training-set MSE.

The Beta weighting function:
$$w_k = \frac{(k/K)^{\theta_1-1}(1-k/K)^{\theta_2-1}}{B(\theta_1,\theta_2)}$$

replaces the K free OLS coefficients with just 2 parameters,
enforcing a smooth lag decay.


In [ ]:
fit_restricted_midas <- function(y, lag_mat, weight_fn,
                                  start=c(1, 5), lower=c(0.1, 0.1)) {
  k <- seq_len(ncol(lag_mat))
  obj <- function(theta) {
    w  <- weight_fn(k, theta)
    xw <- lag_mat %*% w                # weighted single predictor
    X  <- cbind(1, xw)
    b  <- tryCatch(solve(crossprod(X), crossprod(X, y)), error=function(e) NULL)
    if (is.null(b)) return(1e9)
    sum((y - X %*% b)^2)
  }
  opt <- optim(start, obj, method="L-BFGS-B", lower=lower,
               control=list(maxit=500, factr=1e7))
  w_hat <- weight_fn(k, opt$par)
  list(theta=opt$par, weights=w_hat, converged=(opt$convergence==0))
}

predict_restricted_midas <- function(lag_train, lag_test, y_train,
                                      weight_fn, start=c(1,5)) {
  fit  <- fit_restricted_midas(y_train, lag_train, weight_fn, start)
  xw_tr <- lag_train %*% fit$weights
  xw_te <- lag_test  %*% fit$weights
  X_tr  <- cbind(1, xw_tr)
  X_te  <- cbind(1, xw_te)
  b     <- solve(crossprod(X_tr), crossprod(X_tr, y_train))
  pred  <- X_te %*% b
  list(pred=as.vector(pred), theta=fit$theta, weights=fit$weights,
       converged=fit$converged)
}

# Fit Beta-MIDAS and Almon-MIDAS for each macro variable
results_single <- list()
weight_results <- list()

for (var in MACRO_VARS) {
  ltr <- lag_train[[var]][ok_train, , drop=FALSE]
  lte <- lag_test[[var]][ok_test,  , drop=FALSE]
  ytr <- y_train[ok_train]
  yte <- y_test[ok_test]

  # Beta
  beta_fit  <- predict_restricted_midas(ltr, lte, ytr, nbeta_w)
  res_b <- eval_model(paste0("Beta-MIDAS (", var, ")"), yte, beta_fit$pred)
  results_single[[paste0("beta_", var)]] <- res_b
  weight_results[[paste0("beta_", var)]] <- list(
    var=var, type="Beta", theta=beta_fit$theta,
    weights=beta_fit$weights, converged=beta_fit$converged)

  # Almon
  almon_fit <- predict_restricted_midas(ltr, lte, ytr, nealmon_w, start=c(0,0))
  res_a <- eval_model(paste0("Almon-MIDAS (", var, ")"), yte, almon_fit$pred)
  results_single[[paste0("almon_", var)]] <- res_a
  weight_results[[paste0("almon_", var)]] <- list(
    var=var, type="Almon", theta=almon_fit$theta,
    weights=almon_fit$weights, converged=almon_fit$converged)
}


## 7. Visualise fitted weight profiles

The shape of the estimated Beta/Almon polynomial tells you *how far back*
the model looks for each macro variable:
- A weight concentrated on lag 1 → only the most recent month matters
- A hump at lag 2–3 → the model learns a 2–3 month look-back
- A flat profile → all lags equally important (similar to U-MIDAS)


In [ ]:
w_df <- do.call(rbind, lapply(names(weight_results), function(nm) {
  wr <- weight_results[[nm]]
  data.frame(name=nm, var=wr$var, type=wr$type,
             lag=seq_along(wr$weights), weight=wr$weights)
}))

ggplot(w_df, aes(x=lag, y=weight, colour=type)) +
  geom_line(linewidth=0.8) +
  geom_point(size=2) +
  facet_wrap(~var, ncol=2) +
  labs(title="Fitted MIDAS lag weight profiles by macro variable",
       x="Monthly lag (1 = most recent)", y="Weight", colour="Weight type") +
  theme_minimal(base_size=11)


## 8. Multi-predictor Beta-MIDAS

All macro variables enter jointly, each with its own Beta polynomial.
The optimisation finds 2 shape parameters per variable (8 total for 4 variables)
that minimise joint training MSE, then solves for the OLS intercept and
4 scale coefficients.


In [ ]:
fit_multi_beta <- function(y, lag_list, weight_fn, start_theta=NULL) {
  vars   <- names(lag_list)
  n_vars <- length(vars)
  if (is.null(start_theta)) start_theta <- rep(c(1, 5), n_vars)
  k_list <- lapply(lag_list, function(m) seq_len(ncol(m)))

  obj <- function(theta_vec) {
    cols <- list()
    for (j in seq_len(n_vars)) {
      th  <- theta_vec[(2*j-1):(2*j)]
      w   <- weight_fn(k_list[[j]], pmax(th, 0.1))
      cols[[j]] <- lag_list[[j]] %*% w
    }
    X <- cbind(1, do.call(cbind, cols))
    b <- tryCatch(solve(crossprod(X), crossprod(X, y)), error=function(e) NULL)
    if (is.null(b)) return(1e9)
    sum((y - X %*% b)^2)
  }

  opt <- optim(start_theta, obj, method="L-BFGS-B",
               lower=rep(0.1, 2*n_vars),
               control=list(maxit=1000, factr=1e7))

  # Recover weights and coefficients
  theta_vec <- opt$par
  cols_tr   <- list()
  w_list    <- list()
  for (j in seq_len(n_vars)) {
    th <- pmax(theta_vec[(2*j-1):(2*j)], 0.1)
    w  <- weight_fn(k_list[[j]], th)
    w_list[[vars[j]]] <- w
    cols_tr[[j]] <- lag_list[[j]] %*% w
  }
  X_tr <- cbind(1, do.call(cbind, cols_tr))
  b    <- solve(crossprod(X_tr), crossprod(X_tr, y))

  list(theta=theta_vec, weights=w_list, coefs=b,
       converged=(opt$convergence==0))
}

# Fit on train
ltr_multi <- lapply(MACRO_VARS, function(v) lag_train[[v]][ok_train,,drop=FALSE])
lte_multi <- lapply(MACRO_VARS, function(v) lag_test[[v]][ok_test,,drop=FALSE])
names(ltr_multi) <- names(lte_multi) <- MACRO_VARS

cat("Fitting multi-predictor Beta-MIDAS (optimising 8 shape parameters)...\n")
multi_fit <- fit_multi_beta(y_train[ok_train], ltr_multi, nbeta_w)
cat("Converged:", multi_fit$converged, "\n")

# Predict on test
cols_te <- lapply(MACRO_VARS, function(v) lte_multi[[v]] %*% multi_fit$weights[[v]])
X_te_multi <- cbind(1, do.call(cbind, cols_te))
preds_multi <- as.vector(X_te_multi %*% multi_fit$coefs)

res_multi <- eval_model("Multi-predictor Beta-MIDAS", y_test[ok_test], preds_multi)


## 9. MIDAS + Sentiment (weekly aggregated)

The daily sentiment index is aggregated to weekly frequency (5 trading days),
giving a sentiment signal at an intermediate frequency between daily prices and
monthly macro. This is included as an additional MIDAS predictor with its own
Beta weight function — the principled alternative to simple forward-filling.

Weekly frequency ratio: m ≈ 5 (5 trading days per week).
Number of weekly lags: K = 4 (approximately one month of sentiment history).


In [ ]:
sent_path <- "../data/processed/daily_sentiment.csv"

if (file.exists(sent_path)) {
  sent <- read.csv(sent_path, row.names=1)
  sent$Date <- as.Date(rownames(sent))

  # Aggregate daily sentiment to weekly (rolling 5-day mean, non-overlapping)
  sent <- sent[order(sent$Date), ]
  sent$week <- as.integer(floor(as.integer(sent$Date) / 7))   # ISO-week proxy
  weekly_sent <- sent %>%
    group_by(week) %>%
    summarise(date_end     = max(Date),
              sent_weekly  = mean(sentiment_score, na.rm=TRUE),
              .groups="drop") %>%
    arrange(date_end)

  cat("Weekly sentiment observations:", nrow(weekly_sent), "\n")

  # Build lag matrix (K=4 weekly lags) for train and test
  build_weekly_lags <- function(daily_df, weekly_df, n_lags=4) {
    mat <- matrix(NA_real_, nrow=nrow(daily_df), ncol=n_lags)
    for (i in seq_len(nrow(daily_df))) {
      past <- weekly_df$sent_weekly[weekly_df$date_end <= daily_df$Date[i]]
      past <- na.omit(past)
      if (length(past) >= n_lags)
        mat[i, ] <- rev(tail(past, n_lags))
    }
    colnames(mat) <- paste0("sent_wlag", seq_len(n_lags))
    mat
  }

  sent_lag_train <- build_weekly_lags(train, weekly_sent)
  sent_lag_test  <- build_weekly_lags(test,  weekly_sent)

  ok_s_tr <- ok_train & complete.cases(sent_lag_train)
  ok_s_te <- ok_test  & complete.cases(sent_lag_test)
  cat("Train rows with sentiment:", sum(ok_s_tr),
      " Test rows:", sum(ok_s_te), "\n")

  if (sum(ok_s_tr) > 100 && sum(ok_s_te) > 10) {
    # Multi-predictor: macro + sentiment, each with own Beta weights
    all_lag_tr <- c(lapply(MACRO_VARS, function(v) lag_train[[v]][ok_s_tr,,drop=FALSE]),
                    list(sentiment=sent_lag_train[ok_s_tr,,drop=FALSE]))
    all_lag_te <- c(lapply(MACRO_VARS, function(v) lag_test[[v]][ok_s_te,,drop=FALSE]),
                    list(sentiment=sent_lag_test[ok_s_te,,drop=FALSE]))
    names(all_lag_tr)[seq_along(MACRO_VARS)] <- MACRO_VARS
    names(all_lag_te)[seq_along(MACRO_VARS)] <- MACRO_VARS

    cat("Fitting MIDAS + Sentiment model...\n")
    sent_fit <- fit_multi_beta(y_train[ok_s_tr], all_lag_tr, nbeta_w)
    cat("Converged:", sent_fit$converged, "\n")

    cols_te_s <- lapply(names(all_lag_te),
                        function(v) all_lag_te[[v]] %*% sent_fit$weights[[v]])
    X_te_s     <- cbind(1, do.call(cbind, cols_te_s))
    preds_sent <- as.vector(X_te_s %*% sent_fit$coefs)

    res_sent <- eval_model("Beta-MIDAS + Sentiment", y_test[ok_s_te], preds_sent)
  } else {
    cat("Not enough overlapping rows — skipping MIDAS+Sentiment.\n")
    res_sent <- NULL
  }
} else {
  cat("No sentiment file — run 06_sentiment.ipynb first.\n")
  res_sent <- NULL
}


## 10. Model comparison

In [ ]:
all_results <- rbind(
  res_umidas,
  res_adl,
  do.call(rbind, results_single),
  res_multi,
  if (!is.null(res_sent)) res_sent
)
rownames(all_results) <- NULL
all_results <- all_results[order(all_results$rmse), ]
print(all_results, digits=6)


## 11. Forecast plot — best model vs actual

In [ ]:
best_name <- all_results$model[1]
cat("Best model by RMSE:", best_name, "\n")

# Collect all prediction vectors in a named list for plotting
pred_list <- list(
  "U-MIDAS"                   = preds_umidas,
  "ADL-MIDAS"                 = preds_adl,
  "Multi-predictor Beta-MIDAS"= preds_multi
)

plot_df <- data.frame(
  date   = test$Date[ok_test],
  actual = y_test[ok_test]
)
for (nm in names(pred_list)) {
  p <- pred_list[[nm]]
  if (length(p) == nrow(plot_df)) plot_df[[nm]] <- p
}

plot_long <- pivot_longer(plot_df, -date, names_to="model", values_to="value")
ggplot(plot_long, aes(x=date, y=value, colour=model)) +
  geom_line(linewidth=0.5, alpha=0.8) +
  scale_colour_manual(
    values=c("actual"="black","U-MIDAS"="steelblue",
             "ADL-MIDAS"="darkorange","Multi-predictor Beta-MIDAS"="firebrick")) +
  labs(title="MIDAS forecasts vs actual — test set",
       y="Silver log-return", x=NULL, colour=NULL) +
  theme_minimal(base_size=11)


## 12. Save metrics and predictions

In [ ]:
# Keep the best single model for 07_evaluation.ipynb (lowest RMSE)
best_preds <- switch(all_results$model[1],
  "U-MIDAS"                    = preds_umidas,
  "ADL-MIDAS"                  = preds_adl,
  "Multi-predictor Beta-MIDAS" = preds_multi,
  preds_umidas   # fallback
)
best_actuals <- y_test[ok_test]

write.csv(all_results,
          "../data/processed/metrics_midas.csv", row.names=FALSE)
write.csv(data.frame(actual=best_actuals, predicted=best_preds),
          "../data/processed/preds_umidas.csv",  row.names=FALSE)

cat("Saved metrics_midas.csv and preds_umidas.csv\n")
print(all_results[, c("model","rmse","dir_acc","wda")])
